# Module 02 · Unit 02
# Formalizing Security Properties

| | |
|---|---|
| **Estimated time** | 60–70 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Access Control & Policy Logic \[AC\] · Protocol Verification \[PV\] · Adversarial Enumeration \[AE\] |
| **Refresher** | Module 02 · Unit 01 — Predicates and Quantifiers |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Translate a plain-English security requirement into a formal predicate logic statement
2. Identify the **domain**, **predicate**, and **quantifier structure** of any security property
3. Write the **violation condition** of any formal property using quantifier duality
4. Formalise injection safety, privilege bounds, session integrity, and protocol properties
5. Generate a structured **formal audit report** — findings expressed as witnesses to violation conditions
6. Explain how formal specifications differ from tests and why both are necessary


## 🔣 Symbol Primer

No new symbols in this unit — this is the application unit. All notation carries
over from Module 02 · Unit 01.

| Symbol | Read it as | Reminder |
|---|---|---|
| $\forall x \in D,\; P(x)$ | "for all $x$ in $D$, $P(x)$" | Security guarantee — must hold universally |
| $\exists x \in D,\; P(x)$ | "there exists $x$ in $D$ such that $P(x)$" | Vulnerability — one bad object exists |
| $P(x) \Rightarrow Q(x)$ | "if $P(x)$ then $Q(x)$" | Conditional property — obligation for objects satisfying $P$ |
| $\neg, \wedge, \vee$ | not, and, or | Connectives from Module 01 |
| $\equiv$ | "is equivalent to" | Same truth value in all cases |

> **This unit's goal.** Every informal security requirement — in a threat model,
> a compliance standard, a design document — can be expressed as a predicate logic
> formula. Making that translation explicit forces precision, surfaces ambiguity,
> and produces a violation condition that a monitoring system can act on.

---


## 1 · The Translation Method

Moving from informal English to formal predicate logic is a four-step process.
Practise it on every security requirement you encounter.

**Step 1 — Identify the domain.**
What objects is the claim about? Users, inputs, network packets, API calls,
memory addresses, cryptographic keys, agent tool calls?

**Step 2 — Name the predicates.**
What properties do those objects have? Define each as a function:
$\text{admin}(u)$, $\text{injectable}(s)$, $\text{authenticated}(r)$.

**Step 3 — Determine the quantifier structure.**
Is this a *"for all"* claim (guarantee) or a *"there exists"* claim (vulnerability)?
Are there multiple objects with nested quantifiers?

**Step 4 — Write the violation condition.**
Apply negation duality to derive what a monitoring system should detect:
$\neg(\forall x,\; P(x)) \equiv \exists x,\; \neg P(x)$.

### Example — Translating a Compliance Requirement

**Informal:** *"All privileged operations must be recorded in the audit log."*

- **Domain:** $\mathcal{O}$ — the set of all operations performed in the system
- **Predicates:**
  - $\text{privileged}(o)$ — operation $o$ requires elevated permissions
  - $\text{logged}(o)$ — operation $o$ appears in the audit log
- **Quantifier structure:** $\forall$ — this is a universal guarantee

$$\forall o \in \mathcal{O},\; \text{privileged}(o) \Rightarrow \text{logged}(o)$$

- **Violation condition** (negate, apply duality, use conditional identity
  $\neg(P \Rightarrow Q) \equiv P \wedge \neg Q$):

$$\exists o \in \mathcal{O},\; \text{privileged}(o) \wedge \neg\,\text{logged}(o)$$

*"There is a privileged operation that was not logged."*
This is exactly the alert a SIEM system should raise.


## 2 · Injection Safety

SQL injection, prompt injection, command injection, and LDAP injection all share
the same formal shape: an attacker-controlled string reaches an interpreter that
treats part of that string as executable code.

### Formal Specification

Let $\mathcal{I}$ be the set of all possible input strings to a system.
Define:
- $\text{user\_controlled}(s)$ — $s$ contains data from an external source
- $\text{reaches\_interpreter}(s, c)$ — string $s$ reaches interpreter $c$
  (SQL engine, shell, prompt context) without sanitisation
- $\text{executable}(s, c)$ — $s$ contains syntax that interpreter $c$ would execute

**Injection safety property:**

$$\forall s \in \mathcal{I},\; \forall c \in \mathcal{C},\;
  \text{user\_controlled}(s) \wedge \text{reaches\_interpreter}(s, c)
  \Rightarrow \neg\,\text{executable}(s, c)$$

*"For every input string and every interpreter, if the string is user-controlled
and reaches the interpreter, then it cannot be executed."*

**Violation condition:**

$$\exists s \in \mathcal{I},\; \exists c \in \mathcal{C},\;
  \text{user\_controlled}(s) \wedge \text{reaches\_interpreter}(s, c)
  \wedge\; \text{executable}(s, c)$$

*"There exists a user-controlled string that reaches an interpreter and can
be executed."* This is the existence of an injection vulnerability.

### The Prompt Injection Variant

For large language models, $c$ is the **prompt context** and $\text{executable}(s, c)$
means $s$ causes the model to follow instructions embedded in $s$ rather than the
system prompt. The formal shape is identical. The defender's goal is the same
$\forall s\,\forall c$ property; the attacker's goal is to find one witness to the
$\exists s\,\exists c$ violation.


## 3 · Privilege Bounds

Privilege bound properties say that no principal in the system ever exceeds
a permitted permission level. This is the predicate logic form of the invariant
from Module 00.

### Formal Specification

Let $\mathcal{P}$ be the set of principals (users, processes, agents),
$\mathcal{R}$ the set of resources, and $\mathcal{L}$ a set of permission levels
with a total order $\leq$.

Define:
- $\text{level}(p)$ — the current permission level of principal $p$
- $\text{max\_level}(p)$ — the maximum permitted level for $p$
- $\text{can\_access}(p, r)$ — principal $p$ has access to resource $r$
- $\text{requires\_level}(r)$ — the minimum level required to access $r$

**Privilege bound property:**

$$\forall p \in \mathcal{P},\; \text{level}(p) \leq \text{max\_level}(p)$$

**Least privilege property:**

$$\forall p \in \mathcal{P},\; \forall r \in \mathcal{R},\;
  \text{can\_access}(p, r) \Rightarrow
  \text{level}(p) \geq \text{requires\_level}(r)$$

**No privilege escalation property:**

$$\forall p \in \mathcal{P},\; \forall t_1 < t_2,\;
  \text{level}(p, t_2) \leq \text{level}(p, t_1) + \Delta_{\max}$$

where $t_1, t_2$ are time steps and $\Delta_{\max}$ is the maximum permitted
single-step increase. This bounds how fast privilege can grow — a property
that would have prevented several real-world container escape escalations.

### Connection to Module 00

The privilege bound property is the formal predicate logic statement of the
invariant $I(s) : \text{privilege}(s) \leq \text{MAX}$ from Module 00. The
connection is direct: $\forall p,\; I(p)$ is the universal quantification of
the invariant over all principals. We have the tools now to be precise about
both the property and its violation.


## 4 · Session Integrity

A session integrity property ensures that every action taken within a session
was authorised by the original authenticated principal — no request is processed
on behalf of a user who did not make it.

### Formal Specification

Let $\mathcal{S}$ be the set of sessions, $\mathcal{U}$ the set of users,
$\mathcal{A}$ the set of actions. Define:
- $\text{owner}(s)$ — the authenticated user who created session $s$
- $\text{actor}(a)$ — the user whose credentials were used to perform action $a$
- $\text{session}(a)$ — the session in which action $a$ was performed
- $\text{authorised}(u, a)$ — user $u$ explicitly requested action $a$

**Session binding property:**

$$\forall a \in \mathcal{A},\;
  \text{actor}(a) = \text{owner}(\text{session}(a))$$

*"Every action is performed under the credentials of the session owner."*

**CSRF (Cross-Site Request Forgery) vulnerability — the violation:**

$$\exists a \in \mathcal{A},\;
  \text{actor}(a) = \text{owner}(\text{session}(a))
  \;\wedge\; \neg\,\text{authorised}(\text{actor}(a),\, a)$$

*"There exists an action performed with the correct credentials but not
explicitly requested by the user."* The credentials match, but the intent
does not. This is the precise formal definition of CSRF — the session binding
property holds (same credentials), but the authorisation property does not
(the user did not choose to make this request).

This example shows that a single English sentence — "session integrity" —
actually decomposes into two separable formal properties that can be violated
independently.


## 5 · Agentic AI — Tool Call Safety

As AI agents gain the ability to call tools, invoke APIs, and take actions in
the world, the question of which tool calls are permitted becomes a formal
security property. This is the frontier where the mathematics of this course
meets the Agent Scrutiny framework directly.

### Formal Specification

Let $\mathcal{G}$ be the set of agents, $\mathcal{T}$ the set of available tools,
$\mathcal{C}$ the set of call contexts (the state of the agent's task). Define:
- $\text{granted}(g, t)$ — agent $g$ was explicitly granted permission to call tool $t$
- $\text{calls}(g, t, c)$ — agent $g$ invokes tool $t$ in context $c$
- $\text{in\_scope}(c, t)$ — tool call $t$ is within the scope of the current task $c$

**Tool call authorisation property:**

$$\forall g \in \mathcal{G},\; \forall t \in \mathcal{T},\; \forall c \in \mathcal{C},\;
  \text{calls}(g, t, c) \Rightarrow \text{granted}(g, t)$$

*"Every tool call made by every agent was explicitly granted."*

**Scope containment property:**

$$\forall g \in \mathcal{G},\; \forall t \in \mathcal{T},\; \forall c \in \mathcal{C},\;
  \text{calls}(g, t, c) \Rightarrow \text{in\_scope}(c, t)$$

*"Every tool call is within the scope of the current task."*

**Prompt injection violation** — an attacker-controlled input causes the agent
to call a tool it would not have called under the original task:

$$\exists g,\; \exists t,\; \exists c,\; \exists c',\;
  \text{calls}(g, t, c') \wedge \neg\,\text{in\_scope}(c, t)
  \wedge \text{injected}(c', c)$$

where $\text{injected}(c', c)$ means context $c'$ was produced by an attacker
modifying context $c$. The agent calls a tool that is out of scope for the
original task, because injected instructions changed the apparent context.

This is the formal specification underlying Agent Scrutiny's tool call
evaluation criteria — the same criteria you will be working with in your PhD.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AC\] \[PV\] \[AE\]

| Security property | Formal structure | Violation structure |
|---|---|---|
| Injection safety | $\forall s\,\forall c,\; \text{user\_ctrl}(s) \wedge \text{reaches}(s,c) \Rightarrow \neg\,\text{exec}(s,c)$ | $\exists s\,\exists c$ witness |
| Privilege bound | $\forall p,\; \text{level}(p) \leq \text{max}(p)$ | $\exists p$ exceeding bound |
| Audit completeness | $\forall o,\; \text{privileged}(o) \Rightarrow \text{logged}(o)$ | $\exists o$ unlogged |
| Session binding | $\forall a,\; \text{actor}(a) = \text{owner}(\text{session}(a))$ | $\exists a$ with mismatched credentials |
| CSRF safety | $\forall a,\; \text{actor}(a) = \text{owner}(\text{session}(a)) \wedge \text{authorised}(\text{actor}(a), a)$ | $\exists a$ with unintended action |
| Agent tool safety | $\forall g\,\forall t\,\forall c,\; \text{calls}(g,t,c) \Rightarrow \text{granted}(g,t)$ | $\exists g,t,c$ unauthorised call |
| Prompt injection | $\forall g\,\forall c,\; \neg\,\text{injected}(c)$ | $\exists g,c$ where injected context triggers out-of-scope call |

**The pattern is universal.** Every security guarantee is a $\forall$ claim.
Every vulnerability is the $\exists$ witness to its negation. Every security
audit is a search for witnesses. Every formal proof of security is a demonstration
that no such witnesses exist.

The mathematics you have built over Modules 00–02 is now sufficient to write a
formal security specification for any system you can describe. The rest of the
course builds the structures — graphs, automata, number theory — that let you
reason about the *properties of those systems* at scale.

---


## Python: Visualization & Verification

**1 · Formal Property Evaluator** — a reusable engine that takes a formal
security property (domain + predicate + quantifier structure), evaluates it,
and returns structured findings with witnesses.

**2 · Simulated System Audit** — run the evaluator against a simulated system
containing deliberate vulnerabilities; produce a structured audit report with
every finding expressed in formal notation.

**3 · Violation Heatmap** — visualise which system objects violate which
properties across the full audit, making the attack surface visible at a glance.


In [ ]:
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)

print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import itertools

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · Formal Property Evaluator

The engine below models the four-step translation method. A `SecurityProperty`
captures the domain, the predicate, the quantifier type, the formal notation,
and its violation condition. `evaluate` runs the check and returns findings
in a structured format — the building block of an automated security audit.


In [ ]:
# ── 1 · Formal Property Evaluator ────────────────────────────────────────────

class SecurityProperty:
    """
    A formalized security property with:
      name           : short identifier
      formal         : LaTeX-style string of the formal statement
      violation      : LaTeX-style string of the violation condition
      domain         : iterable of objects to check
      predicate      : callable returning True if the property HOLDS for the object
      quantifier     : 'forall' or 'exists'
      description    : plain-English description
    """
    def __init__(self, name, formal, violation, domain,
                 predicate, quantifier, description):
        self.name        = name
        self.formal      = formal
        self.violation   = violation
        self.domain      = list(domain)
        self.predicate   = predicate
        self.quantifier  = quantifier
        self.description = description

    def evaluate(self):
        """
        Returns a dict:
          holds    : bool
          findings : list of objects that are witnesses (exists) or
                     counterexamples (forall)
          severity : 'PASS' | 'FINDING'
        """
        findings = []
        for obj in self.domain:
            result = self.predicate(obj)
            if self.quantifier == 'forall' and not result:
                findings.append(obj)
            elif self.quantifier == 'exists' and result:
                findings.append(obj)

        if self.quantifier == 'forall':
            holds = len(findings) == 0
        else:
            holds = len(findings) > 0

        return {
            'holds'   : holds,
            'findings': findings,
            'severity': 'PASS' if (
                (self.quantifier == 'forall' and holds) or
                (self.quantifier == 'exists' and not holds)
            ) else 'FINDING',
        }

print("SecurityProperty class defined.")
print("Ready to formalise and evaluate security properties.")


**Note on design.** The `SecurityProperty` class encodes the four-step translation
method directly:

- `domain` = Step 1 (the set of objects)
- `predicate` = Step 2 (the Boolean function on each object)
- `quantifier` = Step 3 ($\forall$ or $\exists$)
- `violation` = Step 4 (pre-computed violation condition string)

The `evaluate` method runs the audit. For a $\forall$ property, it searches for
counterexamples. For a $\exists$ property, it searches for witnesses. Findings are
always the specific objects that evidence the result — the audit trail.


### 2 · Simulated System Audit

We define a simulated system with four components:

- **Users** — with attributes matching the Module 01 ABAC policy (admin, active,
  MFA, suspended) plus an `operations` list
- **Operations** — each tagged as privileged or not, logged or not
- **API endpoints** — each either requiring authentication or not
- **Agent tool calls** — each with a granted flag and an in-scope flag

The system contains **deliberate vulnerabilities** — the audit should find them
and express each finding formally.


In [ ]:
# ── 2 · Simulated System Audit ───────────────────────────────────────────────

# ── System definition ─────────────────────────────────────────────────────────
users = [
    {'id': 'u01', 'name': 'Alice',  'admin': True,  'active': True,  'mfa': True,  'suspended': False},
    {'id': 'u02', 'name': 'Bob',    'admin': True,  'active': True,  'mfa': False, 'suspended': False},
    {'id': 'u03', 'name': 'Carol',  'admin': False, 'active': True,  'mfa': True,  'suspended': False},
    {'id': 'u04', 'name': 'Dave',   'admin': False, 'active': True,  'mfa': False, 'suspended': False},
    {'id': 'u05', 'name': 'Grace',  'admin': True,  'active': True,  'mfa': True,  'suspended': True},
]

operations = [
    {'id': 'op01', 'name': 'delete_user',      'privileged': True,  'logged': True},
    {'id': 'op02', 'name': 'export_data',      'privileged': True,  'logged': False},  # BUG
    {'id': 'op03', 'name': 'view_dashboard',   'privileged': False, 'logged': True},
    {'id': 'op04', 'name': 'reset_password',   'privileged': True,  'logged': True},
    {'id': 'op05', 'name': 'modify_acl',       'privileged': True,  'logged': False},  # BUG
    {'id': 'op06', 'name': 'read_public_docs', 'privileged': False, 'logged': False},
]

endpoints = [
    {'id': 'ep01', 'path': '/api/users',       'requires_auth': True},
    {'id': 'ep02', 'path': '/api/admin',        'requires_auth': True},
    {'id': 'ep03', 'path': '/api/health',       'requires_auth': False},
    {'id': 'ep04', 'path': '/api/internal/cfg', 'requires_auth': False},  # BUG
    {'id': 'ep05', 'path': '/api/export',       'requires_auth': True},
]

tool_calls = [
    {'id': 'tc01', 'agent': 'agent_a', 'tool': 'read_file',    'granted': True,  'in_scope': True},
    {'id': 'tc02', 'agent': 'agent_a', 'tool': 'send_email',   'granted': False, 'in_scope': False}, # BUG
    {'id': 'tc03', 'agent': 'agent_b', 'tool': 'query_db',     'granted': True,  'in_scope': True},
    {'id': 'tc04', 'agent': 'agent_b', 'tool': 'delete_record','granted': True,  'in_scope': False}, # BUG
    {'id': 'tc05', 'agent': 'agent_c', 'tool': 'list_files',   'granted': True,  'in_scope': True},
]

# ── Define security properties ────────────────────────────────────────────────
properties = [
    SecurityProperty(
        name        = 'admin_mfa',
        formal      = '∀u ∈ Users, admin(u) → mfa(u)',
        violation   = '∃u ∈ Users, admin(u) ∧ ¬mfa(u)',
        domain      = users,
        predicate   = lambda u: not u['admin'] or u['mfa'],
        quantifier  = 'forall',
        description = 'Every admin has MFA enrolled',
    ),
    SecurityProperty(
        name        = 'no_suspended_admin',
        formal      = '∀u ∈ Users, ¬(admin(u) ∧ suspended(u))',
        violation   = '∃u ∈ Users, admin(u) ∧ suspended(u)',
        domain      = users,
        predicate   = lambda u: not (u['admin'] and u['suspended']),
        quantifier  = 'forall',
        description = 'No admin account is suspended',
    ),
    SecurityProperty(
        name        = 'audit_completeness',
        formal      = '∀o ∈ Ops, privileged(o) → logged(o)',
        violation   = '∃o ∈ Ops, privileged(o) ∧ ¬logged(o)',
        domain      = operations,
        predicate   = lambda o: not o['privileged'] or o['logged'],
        quantifier  = 'forall',
        description = 'Every privileged operation is audit-logged',
    ),
    SecurityProperty(
        name        = 'endpoint_auth',
        formal      = '∀e ∈ Endpoints, internal(e) → requires_auth(e)',
        violation   = '∃e ∈ Endpoints, internal(e) ∧ ¬requires_auth(e)',
        domain      = endpoints,
        predicate   = lambda e: 'internal' not in e['path'] or e['requires_auth'],
        quantifier  = 'forall',
        description = 'Every internal API endpoint requires authentication',
    ),
    SecurityProperty(
        name        = 'tool_authorisation',
        formal      = '∀tc ∈ Calls, calls(tc) → granted(tc)',
        violation   = '∃tc ∈ Calls, calls(tc) ∧ ¬granted(tc)',
        domain      = tool_calls,
        predicate   = lambda tc: tc['granted'],
        quantifier  = 'forall',
        description = 'Every agent tool call was explicitly granted',
    ),
    SecurityProperty(
        name        = 'tool_scope',
        formal      = '∀tc ∈ Calls, calls(tc) → in_scope(tc)',
        violation   = '∃tc ∈ Calls, calls(tc) ∧ ¬in_scope(tc)',
        domain      = tool_calls,
        predicate   = lambda tc: tc['in_scope'],
        quantifier  = 'forall',
        description = 'Every agent tool call is within the current task scope',
    ),
]

# ── Run audit and print report ────────────────────────────────────────────────
results = []
print("=" * 70)
print("FORMAL SECURITY AUDIT REPORT")
print("=" * 70)

for prop in properties:
    result = prop.evaluate()
    results.append((prop, result))
    status  = '✓ PASS   ' if result['severity'] == 'PASS' else '✗ FINDING'
    print(f"[{status}]  {prop.description}")
    print(f"  Property : {prop.formal}")
    if result['findings']:
        print(f"  Violation: {prop.violation}")
        for obj in result['findings']:
            name = obj.get('name') or obj.get('path') or obj.get('name') or f"{obj.get('agent','?')}.{obj.get('tool','?')}" or obj.get('id','?')
            print(f"  Witness  : {name}  (id={obj.get('id','?')})")

finding_count = sum(1 for _, r in results if r['severity'] == 'FINDING')
print(f"{'=' * 70}")
print(f"Summary: {finding_count} finding(s), {len(properties)-finding_count} passed")
print("=" * 70)


**What do you see?**

The audit report surfaces four findings — each with a formal statement, a
violation condition, and a specific witness:

- **Admin MFA** — Bob is an admin without MFA. The witness is Bob.
- **No suspended admin** — Grace is a suspended admin. The witness is Grace.
- **Audit completeness** — `export_data` and `modify_acl` are privileged but
  unlogged. Two witnesses.
- **Internal endpoint auth** — `/api/internal/cfg` is an internal endpoint with
  no authentication requirement. One witness.
- **Tool authorisation** — `agent_a` called `send_email` without a grant.
- **Tool scope** — `agent_b` called `delete_record` outside task scope.

Each finding is expressed in the same format: formal property, formal violation,
specific witness. This is the output format of a rigorous security review — not
"there might be issues with MFA" but "$\exists u \in \text{Users},\; \text{admin}(u)
\wedge \neg\,\text{mfa}(u)$, witness: Bob."


### 3 · Violation Heatmap — The Full Attack Surface

The heatmap below visualises which system objects (rows) violate which security
properties (columns). Green = property holds, Red = violation. Reading across a
row tells you all the problems with one object. Reading down a column tells you
all the objects that violate one property.


In [ ]:
# ── 3 · Violation Heatmap ────────────────────────────────────────────────────

# Collect all objects across domains with their property results
# We evaluate each property against ALL objects (returns False if not in domain)
all_objects = (
    [(u['name'],       'user',     u) for u in users]       +
    [(o['name'],       'operation',o) for o in operations]   +
    [(e['path'],       'endpoint', e) for e in endpoints]    +
    [(f"{tc['agent']}.{tc['tool']}", 'tool_call', tc) for tc in tool_calls]
)

prop_labels = [p.name.replace('_', ' ') for p in properties]
obj_labels  = [name for name, _, _ in all_objects]

# For each object, evaluate each property (True = holds, False = violation)
# If the object is not in the property's domain, mark as N/A (0.5)
matrix = np.full((len(all_objects), len(properties)), 0.5)

for j, prop in enumerate(properties):
    domain_ids = {id(obj) for obj in prop.domain}
    for i, (name, obj_type, obj) in enumerate(all_objects):
        if id(obj) in domain_ids:
            matrix[i, j] = float(prop.predicate(obj))

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 8))
cmap = ListedColormap([TS_RED, '#e8e8e8', TS_GREEN])
ax.imshow(matrix, cmap=cmap, vmin=0, vmax=1, aspect='auto')

for r in range(len(all_objects)):
    for c in range(len(properties)):
        val = matrix[r, c]
        if val == 0.5:
            txt, color = '·', TS_GRAY
        elif val == 1.0:
            txt, color = '✓', 'white'
        else:
            txt, color = '✗', 'white'
        ax.text(c, r, txt, ha='center', va='center',
                fontsize=11, fontweight='bold', color=color)

# Type labels on left
type_colors = {'user': TS_BLUE, 'operation': TS_AMBER,
               'endpoint': TS_GREEN, 'tool_call': '#8e44ad'}
for i, (name, obj_type, _) in enumerate(all_objects):
    ax.text(-0.6, i, f'[{obj_type[:4]}]', ha='right', va='center',
            fontsize=7, color=type_colors[obj_type])

ax.set_xticks(range(len(prop_labels)))
ax.set_xticklabels(prop_labels, rotation=25, ha='right', fontsize=9)
ax.set_yticks(range(len(obj_labels)))
ax.set_yticklabels(obj_labels, fontsize=9)
ax.set_title('Formal Audit — Violation Heatmap'
             '✓ Property holds   ✗ Violation found   · Not in domain',
             pad=12, fontsize=11)
ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

handles = [
    mpatches.Patch(color=TS_GREEN,  label='✓ Property holds'),
    mpatches.Patch(color=TS_RED,    label='✗ Violation'),
    mpatches.Patch(color='#e8e8e8', label='· Not in domain'),
]
ax.legend(handles=handles, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

# Count violations by column (property) and by row (object)
print("Violations by property:")
for j, prop in enumerate(properties):
    n = sum(1 for i in range(len(all_objects))
            if matrix[i,j] == 0.0)
    if n: print(f"  {prop.name:<25} {n} violation(s)")

print()
print("Objects with violations:")
for i, (name, obj_type, _) in enumerate(all_objects):
    violations = [properties[j].name for j in range(len(properties))
                  if matrix[i,j] == 0.0]
    if violations:
        print(f"  {name:<30} violates: {', '.join(violations)}")


**What do you see?**

The heatmap gives a complete picture of the attack surface in one view:

- **Rows** are system objects — grey dots (·) mean the object is not in that
  property's domain (an operation is not evaluated against a user property).
- **Columns** are security properties — a column full of green means the property
  holds universally; any red cell is a violation with a specific witness.
- **Bob and Grace** (top two users) each have a red cell in different columns —
  different failure modes, both surfaced by the formal audit.
- **`export_data` and `modify_acl`** are red in the audit completeness column.
- The agent tool call rows show two distinct failure modes: unauthorised call
  (tool authorisation) vs out-of-scope call (tool scope) — the formal properties
  decomposed what might look like "one AI safety issue" into two separable,
  independently testable properties.

This is what a formal security specification buys you: **precision about what
the properties are, what objects they apply to, and what the findings mean.**


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. ADD A NEW PROPERTY
#    Formalise and add: "Every active user must have completed security training."
#    Add a 'training_complete' attribute to the users list (set some to False).
#    Write the SecurityProperty, run evaluate(), add it to the audit.
#    Formal statement:  ∀u ∈ Users, active(u) → training_complete(u)
#    Violation:         ∃u ∈ Users, active(u) ∧ ¬training_complete(u)
#
# 2. NESTED PROPERTY
#    Formalise the two-argument predicate: can_access(user, resource).
#    Define a resource list and an access rule.
#    Write and evaluate:
#      ∀r ∈ Resources, ∃u ∈ Users, can_access(u, r)
#      (Every resource has at least one authorised user)
#    Is this a forall-exists property? How would you implement it
#    using nested calls to your forall/exists functions from Unit 01?
#
# 3. FORMAL SPECIFICATION EXERCISE
#    Take this informal requirement from an AI governance document:
#      "The AI system shall not take actions that were not explicitly
#       requested by an authorised user within the current session."
#    Apply the four-step translation method:
#      Step 1: Identify the domain(s)
#      Step 2: Name the predicates
#      Step 3: Determine quantifier structure
#      Step 4: Write the violation condition
#    Compare your formalisation to the agent tool call safety property
#    in Section 5. Are they equivalent? What does each capture that
#    the other might miss?

# Your work here:


---

## Summary

| Concept | Definition | Security use |
|---|---|---|
| **Translation method** | Domain → predicates → quantifier → violation | Turn any requirement into a testable formal claim |
| **$\forall$ property** | A guarantee over all objects in a domain | Security invariant — must hold universally |
| **$\exists$ violation** | Witness to the negation of a $\forall$ property | The audit finding — specific object, specific failure |
| **Conditional property** | $\forall x,\; P(x) \Rightarrow Q(x)$ | Obligation: if condition holds, guarantee must follow |
| **Violation of conditional** | $\exists x,\; P(x) \wedge \neg Q(x)$ | One object meeting the condition but not the guarantee |
| **Formal audit** | Evaluate properties against a domain; collect witnesses | Structured security review with formal findings |

---

## Module 02 Complete

| Unit | Content | Payoff |
|---|---|---|
| **01** | Predicates, quantifiers, negation duality, nested quantifiers | The formal vocabulary of security claims |
| **02** | Formalising injection, privilege, session, agent safety | A complete formal specification and audit methodology |

Module 02 closes the logic arc of the course — Modules 00–02 together give you
the tools to state, prove, disprove, and audit any security property that can be
expressed over a domain of objects. The rest of the course builds the mathematical
structures those objects live in.

---

## Up Next

**Module 03 — Sets, Relations & Functions**

The domains and predicates we have been using informally get rigorous foundations.
Sets give us the language to describe collections of objects precisely. Relations
formalise the two-argument predicates ($\text{can\_access}(u, r)$, $\text{can\_decrypt}(u, k)$).
Functions — especially bijections — underpin type systems, input validation, and
the permutations at the heart of cryptographic ciphers.

$\rightarrow$ `modules/module-03/unit-01-sets-operations.ipynb`
